# 06 — Train aggregators (Phase 2, Step 11)

Обучаем 4 групповых агрегатора поверх замороженного SASRec на одном и том же split групп. Артефакты входа (Phase 1 чекпоинт + кэш скоров + аудио + user profiles) подгружаются с HF-датасета `Vladislavbro-500/music-recommendations`.

**Что делает ноутбук:**
1. Скачивает артефакты из HF в `artifacts/`.
2. Строит общий train/val split групп (10k / 2k, seed=42).
3. Тренирует AGREE → GroupIM → AudioAGREE → GroupCrossAttention. Конфиг каждого — в своей ячейке.
4. Сводная таблица + графики val NDCG@10 по эпохам.

**Test-eval не делаем здесь** — это шаг 12 (`07_eval_groups.ipynb`), чтобы не подсматривать в test при подборе конфигов.

In [ ]:
# Colab bootstrap (раскомментировать в Colab):
# from google.colab import userdata
# token = userdata.get('git')
# !git clone -b models-1 https://$token@github.com/Vladislavbro/music-recommendations.git
# %cd music-recommendations
# !pip install -q datasets pyarrow numpy pandas torch huggingface_hub

In [ ]:
import os, sys, json, pickle, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print('project root:', PROJECT_ROOT)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

In [ ]:
# Подгружаем артефакты из HF-датасета `Vladislavbro-500/music-recommendations`.
# Локально (Mac) файлы уже есть — hf_hub_download просто вернёт путь из кэша/диска.
# На Colab скачает в artifacts/.
from huggingface_hub import hf_hub_download

HF_REPO = 'Vladislavbro-500/music-recommendations'
HF_REPO_TYPE = 'dataset'
ARTIFACTS = PROJECT_ROOT / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

needed = [
    'gsasrec/item_id_to_idx.pkl',
    'user_scores_cache/scores.parquet',
    'audio/embeddings.npy',
    'audio/user_profiles.npy',
    'audio/uid_to_row.pkl',
    'audio/user_audio_valid.npy',
]
for rel in needed:
    dst = ARTIFACTS / rel
    if dst.exists():
        print(f'  [skip]  {rel}  ({dst.stat().st_size / 2**20:.1f} MB)')
        continue
    print(f'  [pull]  {rel} ...', flush=True)
    p = hf_hub_download(
        repo_id=HF_REPO,
        repo_type=HF_REPO_TYPE,
        filename=rel,
        local_dir=str(ARTIFACTS),
    )
    print(f'           -> {p}')

## 1. Shared setup

Здесь грузится всё, что не зависит от выбора агрегатора: данные, кэш скоров, аудио, синтез групп. Эти ячейки прогоняются один раз; ячейки методов (ниже) дёргают только подготовленные переменные.

In [ ]:
# Загрузка YAMBDA-50m + GTS — повторяем протокол Phase 1, чтобы train/val/test
# совпали по timestamp-ам с тем, на чём учился скорер.
from src.data.yambda_loader import (
    load_yambda, filter_listens, filter_min_popularity, apply_item_remap,
)
from src.data.splits import global_temporal_split, SplitConfig

HF_CACHE = os.environ.get('HF_DATASETS_CACHE', None)
raw = load_yambda('50m', cache_dir=HF_CACHE)['interactions']
df = filter_listens(raw)
df = filter_min_popularity(df, min_count=5)

with open(ARTIFACTS / 'gsasrec' / 'item_id_to_idx.pkl', 'rb') as f:
    item_id_to_idx = pickle.load(f)
df = apply_item_remap(df, item_id_to_idx)
assert df['item_idx'].isna().sum() == 0
df['item_idx'] = df['item_idx'].astype('int64')
n_items = max(item_id_to_idx.values())

train_df, val_df, test_df = global_temporal_split(df, SplitConfig())
print(f'events: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}')
print(f'users: train={train_df["uid"].nunique():,}  val={val_df["uid"].nunique():,}  test={test_df["uid"].nunique():,}')
print(f'n_items (from item_id_to_idx): {n_items:,}')

In [ ]:
# Per-user score cache → user_topk (для построения C_G) и user_score_lookup
# (для per_user_scores в forward агрегатора).
from src.eval.group_eval import topk_from_score_cache, test_targets_from_df
from src.training.group_trainer import build_user_score_lookup

scores_df = pd.read_parquet(ARTIFACTS / 'user_scores_cache' / 'scores.parquet')
print('scores rows:', len(scores_df), '| uids:', scores_df['uid'].nunique(),
      '| K per uid:', int(scores_df.groupby('uid').size().iloc[0]))

t0 = time.time()
user_topk = topk_from_score_cache(scores_df)
user_score_lookup = build_user_score_lookup(scores_df)
print(f'built user_topk + score_lookup in {time.time()-t0:.1f}s, n_users={len(user_topk):,}')

In [ ]:
# Audio артефакты. ID-методам (AGREE, GroupIM) аудио не нужно, но Trainer
# принимает один и тот же набор тензоров — они их просто игнорируют.
item_audio = np.load(ARTIFACTS / 'audio' / 'embeddings.npy')
user_profiles = np.load(ARTIFACTS / 'audio' / 'user_profiles.npy')
with open(ARTIFACTS / 'audio' / 'uid_to_row.pkl', 'rb') as f:
    uid_to_row = pickle.load(f)

audio_valid_items = np.linalg.norm(item_audio, axis=1) > 0
print(f'item_audio: {item_audio.shape}, valid {int(audio_valid_items[1:].sum()):,}/{item_audio.shape[0]-1:,} '
      f'({100*audio_valid_items[1:].mean():.2f}%)')
print(f'user_profiles: {user_profiles.shape}, uid_to_row: {len(uid_to_row):,} entries')

In [ ]:
# Per-user targets: train listens → ground-truth для train-групп,
# val listens → для val-групп. Test-таргеты НЕ собираем (шаг 12).
user_train_targets = test_targets_from_df(train_df)
user_val_targets = test_targets_from_df(val_df)
print(f'users with train listens: {len(user_train_targets):,}')
print(f'users with val   listens: {len(user_val_targets):,}')

In [ ]:
# Синтез групп. Pool — юзеры из кэша скоров (= train users минус <2-event filter).
# Размеры по литературе AGREE/GroupIM. Один seed → один и тот же split групп
# для всех 4 методов, чтобы сравнение было honest.
from src.data.group_synthesis import synthesize_random_groups

GROUP_SEED = 42
N_TRAIN_GROUPS = 10_000
N_VAL_GROUPS = 2_000
N_TEST_GROUPS = 2_000  # понадобятся на шаге 12; синтезируем здесь, чтобы зафиксировать seed-split
SIZE_DIST = {2: 0.3, 3: 0.4, 4: 0.2, 5: 0.1}

user_pool = sorted(user_topk.keys())
rng = np.random.default_rng(GROUP_SEED)
train_groups = synthesize_random_groups(user_pool, N_TRAIN_GROUPS, SIZE_DIST, seed=int(rng.integers(1 << 30)))
val_groups   = synthesize_random_groups(user_pool, N_VAL_GROUPS,   SIZE_DIST, seed=int(rng.integers(1 << 30)))
test_groups  = synthesize_random_groups(user_pool, N_TEST_GROUPS,  SIZE_DIST, seed=int(rng.integers(1 << 30)))
print(f'pool={len(user_pool):,}  train_groups={len(train_groups)}  val_groups={len(val_groups)}  test_groups={len(test_groups)}')

In [ ]:
# Сборка GroupSample для train (target = train listens) и val (target = val listens).
# Test-samples собирать здесь не будем — но сами test_groups сохраним для шага 12.
from src.eval.group_eval import build_group_samples

train_samples, train_stats = build_group_samples(
    train_groups, user_topk, user_train_targets,
    ground_truth='union', drop_empty=True, drop_missing_member=True,
)
val_samples, val_stats = build_group_samples(
    val_groups, user_topk, user_val_targets,
    ground_truth='union', drop_empty=True, drop_missing_member=True,
)
print('TRAIN:', {k: train_stats[k] for k in ['n_input_groups','n_kept','n_dropped_empty_targets','candidate_size_mean','target_size_mean','by_size_counts']})
print('VAL  :', {k: val_stats[k] for k in ['n_input_groups','n_kept','n_dropped_empty_targets','candidate_size_mean','target_size_mean','by_size_counts']})

# Сохраняем split групп — пригодится шагу 12 (test_groups в т.ч.).
AGG_DIR = ARTIFACTS / 'aggregators'
AGG_DIR.mkdir(parents=True, exist_ok=True)
with open(AGG_DIR / 'groups_split.pkl', 'wb') as f:
    pickle.dump({
        'train_groups': train_groups,
        'val_groups': val_groups,
        'test_groups': test_groups,
        'group_seed': GROUP_SEED,
        'size_dist': SIZE_DIST,
        'train_stats': train_stats,
        'val_stats': val_stats,
    }, f)
print(f'saved {AGG_DIR / "groups_split.pkl"}')

In [ ]:
# Popularity-веса для negative sampling в trainer'е.
from src.training.group_trainer import compute_pop_counts

pop_counts = compute_pop_counts(train_df, n_items=n_items, item_col='item_idx', smoothing=0.75)
print(f'pop_counts: shape={pop_counts.shape}, sum={pop_counts.sum():.3f}, max={pop_counts.max():.6f}')

## 2. Training

Общая часть — `GroupTrainConfig` с дефолтами + хелпер `make_trainer`. Каждый метод дальше переопределяет только то, что отличается.

**Дефолты:** Adam lr=1e-3, batch=64, n_epochs=20, n_neg=4, patience=5, eval_k=(10, 20).

In [ ]:
from dataclasses import replace
from src.training.group_trainer import GroupTrainConfig, GroupAggregatorTrainer
from src.aggregators import IDBasedAGREE, GroupIM, AudioAGREE, GroupCrossAttention

BASE_CFG = GroupTrainConfig(
    n_epochs=20,
    batch_size=64,
    eval_batch_size=128,
    lr=1e-3,
    weight_decay=0.0,
    n_neg_per_pos=4,
    eval_k=(10, 20),
    early_stop_patience=5,
    seed=42,
    log_every_steps=50,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

def make_trainer(model, out_subdir, **cfg_overrides):
    cfg = replace(BASE_CFG, out_dir=str(AGG_DIR / out_subdir), **cfg_overrides)
    return GroupAggregatorTrainer(
        aggregator=model,
        cfg=cfg,
        user_score_lookup=user_score_lookup,
        pop_counts=pop_counts,
        item_audio=item_audio,
        user_profiles=user_profiles,
        uid_to_row=uid_to_row,
    )

print('base cfg device:', BASE_CFG.device)

### 2.1 ID-based AGREE

Item-aware attention `α_{u,i} = softmax(h^T tanh(W [e_u; e_i]))` на learnable ID-эмбеддингах. Группа: `s_G(i) = Σ_u α_{u,i} · s_{u,i}`.

In [ ]:
torch.manual_seed(BASE_CFG.seed)
model_agree = IDBasedAGREE(
    uid_list=user_pool,
    num_items=n_items,
    d_emb=32,
    d_att=32,
)
trainer_agree = make_trainer(model_agree, 'agree')
n_params = sum(p.numel() for p in model_agree.parameters() if p.requires_grad)
print(f'AGREE params: {n_params:,}')

result_agree = trainer_agree.fit(train_samples, val_samples, verbose=True)
print('AGREE done:', {k: v for k, v in result_agree.items() if k != 'history'})

### 2.2 GroupIM

Item-agnostic attention + MI-регуляризация. `reg_loss_weight=0.5` — середина сетки `{0.1, 0.5, 1.0}` из открытых вопросов лога. Если основная таблица покажет, что GroupIM сильно проигрывает — попробуем другие λ.

In [ ]:
torch.manual_seed(BASE_CFG.seed)
model_groupim = GroupIM(
    uid_list=user_pool,
    num_items=n_items,
    d_emb=32,
    d_att=32,
)
trainer_groupim = make_trainer(model_groupim, 'groupim', reg_loss_weight=0.5)
n_params = sum(p.numel() for p in model_groupim.parameters() if p.requires_grad)
print(f'GroupIM params: {n_params:,}, λ_MI=0.5')

result_groupim = trainer_groupim.fit(train_samples, val_samples, verbose=True)
print('GroupIM done:', {k: v for k, v in result_groupim.items() if k != 'history'})

### 2.3 AudioAGREE

Прямой аналог AGREE на аудио: `α_{u,i} = softmax(φ([ā_u; a_i]))`, φ — 2-слойный MLP с GELU. ID-эмбеддингов нет.

In [ ]:
torch.manual_seed(BASE_CFG.seed)
model_aa = AudioAGREE(
    d_audio=128,
    d_att=64,
)
trainer_aa = make_trainer(model_aa, 'audio_agree')
n_params = sum(p.numel() for p in model_aa.parameters() if p.requires_grad)
print(f'AudioAGREE params: {n_params:,}')

result_aa = trainer_aa.fit(train_samples, val_samples, verbose=True)
print('AudioAGREE done:', {k: v for k, v in result_aa.items() if k != 'history'})

### 2.4 GroupCrossAttention

Multi-head scaled dot-product cross-attention с `Q = a_i, K = ā_u`. Логиты усредняются по головам, итог = `Σ_u α_{u,i} · s_{u,i}` (как у всех остальных).

In [ ]:
torch.manual_seed(BASE_CFG.seed)
model_ca = GroupCrossAttention(
    d_audio=128,
    d_model=64,
    n_heads=4,
)
trainer_ca = make_trainer(model_ca, 'group_cross_attn')
n_params = sum(p.numel() for p in model_ca.parameters() if p.requires_grad)
print(f'GroupCrossAttention params: {n_params:,}')

result_ca = trainer_ca.fit(train_samples, val_samples, verbose=True)
print('GroupCrossAttention done:', {k: v for k, v in result_ca.items() if k != 'history'})

## 3. Comparison (val only)

Test трогать не будем (это шаг 12). Здесь — сравнение **по val** + кривые обучения.

In [ ]:
# Сводная таблица: best val NDCG@10/20 per method (читаем metrics.csv).
rows = []
for name, subdir in [
    ('AGREE',           'agree'),
    ('GroupIM',         'groupim'),
    ('AudioAGREE',      'audio_agree'),
    ('GroupCrossAttn',  'group_cross_attn'),
]:
    mp = AGG_DIR / subdir / 'metrics.csv'
    if not mp.exists():
        rows.append({'method': name, 'best_epoch': None, 'val_NDCG@10': None})
        continue
    m = pd.read_csv(mp)
    best_row = m.iloc[m['val_NDCG@10'].idxmax()]
    rows.append({
        'method': name,
        'best_epoch': int(best_row['epoch']),
        'val_NDCG@10': float(best_row['val_NDCG@10']),
        'epochs_run': len(m),
    })
summary = pd.DataFrame(rows).sort_values('val_NDCG@10', ascending=False)
print(summary.to_string(index=False))

In [ ]:
# Доп. таблица: финальный val NDCG@20 (пересчитываем через trainer.evaluate на best.pt).
from src.eval.group_eval import evaluate_aggregator_scores

def reload_and_eval(model, trainer, ckpt_path):
    sd = torch.load(ckpt_path, map_location=trainer.device)
    model.load_state_dict(sd['aggregator_state'])
    return trainer.evaluate(val_samples)

final = []
for name, model, trainer in [
    ('AGREE',           model_agree,   trainer_agree),
    ('GroupIM',         model_groupim, trainer_groupim),
    ('AudioAGREE',      model_aa,      trainer_aa),
    ('GroupCrossAttn',  model_ca,      trainer_ca),
]:
    ckpt = Path(trainer.cfg.out_dir) / 'best.pt'
    if not ckpt.exists():
        continue
    m = reload_and_eval(model, trainer, ckpt)
    final.append({
        'method': name,
        'val_NDCG@10': m['NDCG@10'],
        'val_NDCG@20': m['NDCG@20'],
        'val_NDCG@10_std': m['NDCG@10_std'],
        **{f'NDCG@10[size={s}]': m['by_size'][s]['NDCG@10'] for s in sorted(m['by_size'])},
    })
final_df = pd.DataFrame(final).sort_values('val_NDCG@10', ascending=False)
print(final_df.to_string(index=False))

In [ ]:
# График train loss + val NDCG@10 по эпохам для всех 4 методов.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
methods = [
    ('AGREE',           'agree'),
    ('GroupIM',         'groupim'),
    ('AudioAGREE',      'audio_agree'),
    ('GroupCrossAttn',  'group_cross_attn'),
]
for name, subdir in methods:
    mp = AGG_DIR / subdir / 'metrics.csv'
    if not mp.exists():
        continue
    m = pd.read_csv(mp)
    axes[0].plot(m['epoch'], m['train_loss'], marker='o', markersize=3, label=name)
    axes[1].plot(m['epoch'], m['val_NDCG@10'], marker='o', markersize=3, label=name)
axes[0].set_title('Train BPR loss'); axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title('Val NDCG@10');    axes[1].set_xlabel('epoch'); axes[1].set_ylabel('NDCG@10'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Что дальше

После прогона на Colab — залить `artifacts/aggregators/` обратно на HF, чтобы шаг 12 видел чекпоинты:

```
hf upload Vladislavbro-500/music-recommendations \
    artifacts/aggregators . \
    --type dataset --commit-message 'phase2 step11: aggregator checkpoints'
```

Шаг 12 (`07_eval_groups.ipynb`) подгрузит `groups_split.pkl` + 4 `best.pt`, посчитает test NDCG@10/20 + bootstrap CI + срез по размеру группы — это и есть финальная таблица для текста ВКР.